In [1]:
# 1) 드라이브에 연결
from google.colab import drive
drive.mount('/content/drive')
# (출력) Mounted at /content/drive

Mounted at /content/drive


In [2]:
# 2) 팀드라이브 내 임베딩 폴더에 접근
import os

EMBEDDING_DIR = "/content/drive/MyDrive/Genshin Build Alchemist (팀)/data/embeddings"

print(os.listdir(EMBEDDING_DIR))

['doc__granite-r2__character.npy', 'doc__granite-r2__character.meta.json', 'doc__granite-r2__character.jsonl', 'doc__granite-r2__weapon-artifact.jsonl', 'doc__granite-r2__weapon-artifact.meta.json', 'doc__granite-r2__weapon-artifact.npy']


In [3]:
# 3) 기존 임베딩에 연결된 JSONL 파일이 어떤 정보를 가지고 있는지 파악하기 위해
import json
from pathlib import Path
from pprint import pprint

# 기존 임베딩 파일 경로
jsonl_path = Path(EMBEDDING_DIR) / "doc__granite-r2__weapon-artifact.jsonl"

# 파일 존재 여부 확인
if not jsonl_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {jsonl_path}")

# JSONL 읽기
records = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print("전체 문서 수:", len(records))
print("\n첫 번째 문서의 데이터 구조:")
pprint(records[0])

print("\n두 번째 문서:")
pprint(records[1])

print("\n세 번째 문서:")
pprint(records[2])

전체 문서 수: 319

첫 번째 문서의 데이터 구조:
{'doc_id': 'weapon:absolution'}

두 번째 문서:
{'doc_id': 'weapon:akuoumaru'}

세 번째 문서:
{'doc_id': 'weapon:alleyhunter'}


In [4]:
# 4) 전체 데이터 구조 확인
from collections import Counter

# 전체 레코드의 필드 구성 확인
field_counts = Counter(
    tuple(sorted(record.keys()))
    for record in records
)

print("필드 구성별 문서 수:")
for fields, count in field_counts.items():
    print(f"{fields}: {count}개")

# 무기와 성유물 문서 개수 확인
type_counts = Counter(
    record["doc_id"].split(":")[0]
    for record in records
)

print("\n문서 유형별 개수:")
print(type_counts)

# 마지막 문서 확인
print("\n마지막 문서:")
print(records[-1])

필드 구성별 문서 수:
('doc_id',): 319개

문서 유형별 개수:
Counter({'weapon': 256, 'artifact': 63})

마지막 문서:
{'doc_id': 'artifact:wandererstroupe'}


In [5]:
# 5) npy 데이터 구조 확인
import numpy as np

npy_path = Path(EMBEDDING_DIR) / "doc__granite-r2__weapon-artifact.npy"

# 기존 임베딩을 메모리에 전부 복사하지 않고 읽기
vectors = np.load(npy_path, mmap_mode="r")

print("벡터 배열 형태:", vectors.shape)
print("데이터 타입:", vectors.dtype)
print("차원 수:", vectors.ndim)

print("\nJSONL 문서 수:", len(records))
print("NPY 벡터 수:", vectors.shape[0])

print("\n첫 번째 벡터 앞 5개 값:")
print(vectors[0][:5])

벡터 배열 형태: (319, 768)
데이터 타입: float32
차원 수: 2

JSONL 문서 수: 319
NPY 벡터 수: 319

첫 번째 벡터 앞 5개 값:
[ 0.00152588 -0.02099609  0.03613281  0.01599121  0.02209473]


In [6]:
# 6) META 파일 확인
meta_path = Path(EMBEDDING_DIR) / "doc__granite-r2__weapon-artifact.meta.json"

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("META 데이터 구조:")
print(meta)

META 데이터 구조:
{'model': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'dimension': 768, 'normalize_embeddings': True, 'count': 319, 'kinds': ['weapon', 'artifact'], 'source': 'data/docs/documents_ko.jsonl'}


In [7]:
# 7) 실제 경로와 데이터 연결 확인
from pathlib import Path
import json
from collections import Counter

# documents 원본 경로
documents_path = Path(EMBEDDING_DIR).parent / "documents_ko.jsonl"

print("파일 존재 여부:", documents_path.exists())
print("파일 경로:", documents_path)

# JSONL 읽기
documents = []

with open(documents_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            documents.append(json.loads(line))

# 무기와 성유물만 추출
item_docs = [
    doc for doc in documents
    if doc.get("kind") in ("weapon", "artifact")
]

print("\n전체 문서:", len(documents))
print("아이템 문서:", len(item_docs))
print("아이템 유형:", Counter(doc["kind"] for doc in item_docs))

# 기존 임베딩 문서와 순서 비교
document_ids = [doc["doc_id"] for doc in item_docs]
embedding_ids = [record["doc_id"] for record in records]

print("\n임베딩 문서와 순서 일치:", document_ids == embedding_ids)
print("문서 ID 집합 일치:", set(document_ids) == set(embedding_ids))

print("\n신규 무기 확인:")
pprint([
    doc for doc in item_docs
    if doc["doc_id"] == "weapon:starsealededict"
])

파일 존재 여부: True
파일 경로: /content/drive/MyDrive/Genshin Build Alchemist (팀)/data/documents_ko.jsonl

전체 문서: 1696
아이템 문서: 319
아이템 유형: Counter({'weapon': 256, 'artifact': 63})

임베딩 문서와 순서 일치: True
문서 ID 집합 일치: True

신규 무기 확인:
[{'doc_id': 'weapon:starsealededict',
  'kind': 'weapon',
  'metadata': {'kind': 'weapon',
               'main_stat': '치명타 피해',
               'name': '별빛 계시',
               'rarity': '5성',
               'source': 'genshin-db 8e5badd',
               'weapon': '법구'},
  'text': '별빛 계시 (5성 법구)\n'
          '기초 공격력: 44 / 주옵션: 치명타 피해 19.2%\n'
          '효과 「수성의 궤적」(1재련): 치명타 피해가 24% 증가한다. 원소전투 스킬 발동 시, 장착 캐릭터가 「결정 조화」 '
          '효과를 획득한다: 원소 마스터리가 48pt 증가한다. 지속 시간: 20초. 지속 시간 동안 주변에 있는 파티 내 캐릭터가 '
          '원소 반응 발동 시, 장착 캐릭터가 「횃불의 규칙」 효과를 1스택 획득한다: 원소 마스터리가 추가로 24pt 증가한다. '
          '해당 효과는 최대 3스택까지 중첩되며, 「결정 조화」 효과가 끝나거나 원소전투 스킬을 다시 발동할 때까지 지속된다.\n'
          '섬광·별 초전도: 「횃불의 규칙」 1스택당 장착 캐릭터의 별 초전도 반응 피해의 치명타 피해가 추가로 12% 증가한다.\n'
          '장착 캐릭터가 대기 상태일 때도 해당 효과가

In [8]:
# 8) 필터링 기준 데이터 확인
import json
from pathlib import Path
from collections import Counter

# 원본 문서 읽기
documents_path = Path(EMBEDDING_DIR).parent / "documents_ko.jsonl"

with open(documents_path, "r", encoding="utf-8") as f:
    docs = [json.loads(line) for line in f if line.strip()]

weapons = [
    d["metadata"] for d in docs
    if d.get("kind") == "weapon"
]

artifacts = [
    d["metadata"] for d in docs
    if d.get("kind") == "artifact"
]

print("=== 무기 종류 ===")
print(Counter(w.get("weapon") for w in weapons))

print("\n=== 무기 성급 ===")
print(Counter(w.get("rarity") for w in weapons))

print("\n=== 무기 주옵션 ===")
print(Counter(w.get("main_stat") or "정보 없음" for w in weapons))

print("\n=== 성유물 성급 ===")
print(Counter(a.get("rarity") for a in artifacts))

print("\n=== 필수 메타데이터 누락 ===")
print({
    "종류": sum(not w.get("weapon") for w in weapons),
    "성급": sum(not w.get("rarity") for w in weapons),
    "주옵션": sum(not w.get("main_stat") for w in weapons)
})

=== 무기 종류 ===
Counter({'한손검': 62, '법구': 56, '활': 50, '양손검': 45, '장병기': 43})

=== 무기 성급 ===
Counter({'4성': 146, '5성': 76, '3성': 24, '1성': 5, '2성': 5})

=== 무기 주옵션 ===
Counter({'공격력': 64, '치명타 확률': 41, '치명타 피해': 38, '원소 충전 효율': 38, '원소 마스터리': 30, 'HP': 18, '정보 없음': 10, '물리 피해 보너스': 9, '방어력': 8})

=== 성유물 성급 ===
Counter({'4성/5성': 46, '3성/4성': 14, '1성/2성/3성': 3})

=== 필수 메타데이터 누락 ===
{'종류': 0, '성급': 0, '주옵션': 10}


In [9]:
# 9) 주옵션이 없는 무기 확인
missing_stats = [
    doc for doc in documents
    if doc.get("kind") == "weapon"
    and not doc.get("metadata", {}).get("main_stat")
]

print("주옵션 누락 무기 수:", len(missing_stats))

print("\n성급별 누락 개수:")
print(Counter(
    doc["metadata"]["rarity"]
    for doc in missing_stats
))

print("\n누락 무기 상세:")
for doc in missing_stats:
    meta = doc["metadata"]
    print(
        meta["name"],
        "|", meta["weapon"],
        "|", meta["rarity"],
        "|", meta.get("main_stat")
    )

주옵션 누락 무기 수: 10

성급별 누락 개수:
Counter({'1성': 5, '2성': 5})

누락 무기 상세:
학도의 노트 | 법구 | 1성 | None
초보자의 장창 | 장병기 | 1성 | None
무인검 | 한손검 | 1성 | None
사냥활 | 활 | 1성 | None
철촉창 | 장병기 | 2성 | None
용병 중검 | 양손검 | 2성 | None
포켓 주술서 | 법구 | 2성 | None
노련의 사냥활 | 활 | 2성 | None
실버 소드 | 한손검 | 2성 | None
훈련용 대검 | 양손검 | 1성 | None


In [10]:
# 10)
import subprocess
from google.colab import userdata
from pathlib import Path

REPO_DIR = Path("/content/genshin-build-alchemist")

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Colab Secrets에서 GITHUB_TOKEN을 확인해 줘.")

url = f"https://{token}@github.com/genshin-build-alchemist/genshin-build-alchemist.git"

r = subprocess.run(
    ["git", "clone", "-q", url, str(REPO_DIR)],
    capture_output=True,
    text=True
)

output = (r.stdout + r.stderr).replace(token, "***")

if r.returncode != 0:
    raise RuntimeError(output)

print("GitHub 저장소 클론 완료")
print("vector.py 존재:", (REPO_DIR / "core" / "vector.py").exists())
print("저장소 경로:", REPO_DIR)

GitHub 저장소 클론 완료
vector.py 존재: True
저장소 경로: /content/genshin-build-alchemist


In [13]:
import sys
from pathlib import Path
sys.path.insert(0, "/content/genshin-build-alchemist")
print(sys.path[0])

/content/genshin-build-alchemist


In [14]:
# 11) GitHub 레포 검색 코드 연결 및 검색 방식 준비

import sys
import json
from pathlib import Path

# ==========================================
# 1. GitHub 레포 경로
# ==========================================

REPO_DIR = Path("/content/genshin-build-alchemist")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

!cd /content/genshin-build-alchemist && git pull origin main
!cd /content/genshin-build-alchemist && git checkout origin/main -- core/aliases.py
!cd /content/genshin-build-alchemist && git checkout rag/item-search-eval

# ==========================================
# 2. 레포의 실제 검색 함수 불러오기
# ==========================================

from core.vector import load_item_index, item_vector_search
from core.retrieval import item_filters
from core.kb import load_kb
from core.router import Router

# ==========================================
# 3. 실제 아이템 검색 인덱스 로드
# ==========================================

load_result = load_item_index(EMBEDDING_DIR)

print("인덱스 로드:", load_result)

if not load_result:
    raise RuntimeError("아이템 검색 인덱스 로드 실패")

# ==========================================
# 4. Knowledge Base 및 이름·별명 검색 준비
# ==========================================

kb = load_kb()
router = Router(kb)

# 원본 문서 메타데이터
DOCS_PATH = Path(EMBEDDING_DIR).parent / "documents_ko.jsonl"

documents = [
    json.loads(line)
    for line in DOCS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

metadata_by_id = {
    doc["doc_id"]: doc["metadata"]
    for doc in documents
    if doc.get("kind") in ("weapon", "artifact")
}

print("무기·성유물 메타데이터:", len(metadata_by_id))

# ==========================================
# 5. 순수 벡터 검색
# ==========================================

def vector_search(question):
    """레포의 필터링을 적용한 벡터 검색"""

    return item_vector_search(
        question,
        k=5,
        **item_filters(question)
    )


# ==========================================
# 6. 이름·별명 검색
# ==========================================

def name_search(question):
    """정식 이름 또는 별명을 찾아 정식 아이템 ID 반환"""

    item_name, kind = router.find_item(question)

    if item_name is None:
        return []

    item_table = (
        kb.weapon_by_name
        if kind == "weapon"
        else kb.artifact_by_name
    )

    info = item_table.get(item_name)

    if info is None:
        return []

    return [f"{kind}:{info['key']}"]


# ==========================================
# 7. 실제 챗봇 검색 방식
# ==========================================

def chatbot_search(question):
    """
    이름·별명이 있으면 이름 검색
    없으면 벡터 검색
    """

    name_result = name_search(question)

    if name_result:
        return name_result

    return vector_search(question)


# ==========================================
# 8. 검색 방식 준비 확인
# ==========================================

print("\n검색 방식 준비 완료")
print("1. 벡터 검색")
print("2. 이름·별명 우선 검색 → 벡터 검색")
print("인덱스 문서 수:", len(metadata_by_id))

From https://github.com/genshin-build-alchemist/genshin-build-alchemist
 * branch            main       -> FETCH_HEAD
Already up to date.
branch 'rag/item-search-eval' set up to track 'origin/rag/item-search-eval'.
Switched to a new branch 'rag/item-search-eval'


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/21.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

인덱스 로드: True
무기·성유물 메타데이터: 319

검색 방식 준비 완료
1. 벡터 검색
2. 이름·별명 우선 검색 → 벡터 검색
인덱스 문서 수: 319


In [ ]:
for a in kb.artifacts:
    if "현인" in a.get("name","") or "누각" in a.get("name",""):
        print(a["key"], a["name"])

In [ ]:
# 12) 레포 검색 함수 동작 확인 (필터적용 확인, 기능점검)

# 1. 벡터 검색 테스트
sample_question = "치명타 확률이 주옵션인 한손검 추천"

filters = item_filters(sample_question)

vector_result = item_vector_search(
    sample_question,
    k=5,
    **filters
)

print("적용 필터:", filters)
print("\n[벡터 검색 결과]")

for rank, doc_id in enumerate(vector_result, 1):
    meta = metadata_by_id.get(doc_id, {})
    print(
        f"{rank}위 | {doc_id} | "
        f"{meta.get('rarity', '-')} | "
        f"{meta.get('weapon', '-')} | "
        f"{meta.get('main_stat', '-')}"
    )

print("검색 개수:", len(vector_result))


# 2. 이름·별명 검색 테스트
name_question = "안개를 가르는 회광의 무기 효과를 알려줘."

print("\n[이름 검색 결과]")
print(name_search(name_question))


# 3. 실제 챗봇 방식 테스트
print("\n[챗봇 방식 결과]")
print("이름이 있는 질문:", chatbot_search(name_question))
print("이름이 없는 질문:", chatbot_search(sample_question))

In [16]:
# 13) 최종 평가 (레포 코드 기준)
# 정식 이름 / 별명 / 효과 × 무기·성유물 + 챗봇 전체

import pandas as pd
from collections import defaultdict
import re
import numpy as np

# ============================================================
# 1. 평가 질문 만들기
# ============================================================

def is_high_rarity(meta):
    rarity = str(meta.get("rarity", ""))
    stars = set(re.findall(r"(?<!\d)([1-5])\s*성", rarity))
    return bool(stars & {"4", "5"})

def unique_and_sample(items, n=12):
    seen = set()
    unique = []
    for doc_id, meta, kind in items:
        name = re.sub(r"[\W_]+", "", meta["name"]).casefold()
        if name not in seen:
            seen.add(name)
            unique.append((doc_id, meta, kind))
    if len(unique) < n:
        return unique
    idx = np.linspace(0, len(unique)-1, n, dtype=int)
    return [unique[i] for i in idx]

# ----- 정식 이름 질문 (무기 12 + 성유물 12) -----
weapons = [
    (f"weapon:{w['key']}", w, "weapon")
    for w in kb.weapons
    if w.get("key") and w.get("name") and is_high_rarity(w)
]
artifacts = [
    (f"artifact:{a['key']}", a, "artifact")
    for a in kb.artifacts
    if a.get("key") and a.get("name") and is_high_rarity(a)
]

selected_weapons = unique_and_sample(weapons, 12)
selected_artifacts = unique_and_sample(artifacts, 12)

name_questions = []
for i, (doc_id, meta, kind) in enumerate(selected_weapons + selected_artifacts, 1):
    name = meta["name"]
    q = f"{name} 무기 효과 알려줘" if kind == "weapon" else f"{name} 성유물 효과 알려줘"
    name_questions.append({
        "id": f"name-{i:02d}",
        "question": q,
        "gold_docs": [doc_id],
        "kind": kind,
        "qtype": "정식 이름",
    })
print(f"정식 이름 질문: {len(name_questions)}개 (무기 {len(selected_weapons)}, 성유물 {len(selected_artifacts)})")


# ----- 별명 질문 (무기 6 + 성유물 6 예시) -----
# 실제 별명은 프로젝트 aliases 기준으로 나중에 교체 가능
nickname_data = [
    # 무기 별명 (6개)
    ("n-w01", "안개회광 효과 알려줘", "weapon:mistsplitterreforged"),
    ("n-w02", "무공의 검 옵션 뭐야", "weapon:theunforged"),
    ("n-w03", "반암결록 특징 설명해줘", "weapon:primordialjadecutter"),
    ("n-w04", "호마 효과 알려줘", "weapon:staffofhoma"),
    ("n-w05", "천공의 날개 옵션은?", "weapon:skywardharp"),
    ("n-w06", "늑대의 말로 효과 알려줘", "weapon:wolfsgravestone"),
    # 성유물 별명 (6개)
    ("n-a01", "검투사 4세트 효과 뭐야", "artifact:gladiatorsfinale"),
    ("n-a02", "추억의 세트 옵션 알려줘", "artifact:shimenawasreminiscence"),
    ("n-a03", "절연 4세트 설명해줘", "artifact:emblemofseveredfate"),
    ("n-a04", "현인 효과 알려줘", "artifact:lavawalker"),
    ("n-a05", "얼음바람 4세트 뭐야", "artifact:blizzardstrayer"),
    ("n-a06", "누각 세트 옵션 알려줘", "artifact:desertpavilionchronicle"),
]

nickname_questions = [
    {
        "id": eid,
        "question": q,
        "gold_docs": [gold],
        "kind": gold.split(":")[0],
        "qtype": "별명",
    }
    for eid, q, gold in nickname_data
]
print(f"별명 질문: {len(nickname_questions)}개")


# ----- 효과 질문 -----
effect_data = [
    ("w01", "생명의 계약 수치가 늘어날 때 착용자의 피해 보너스도 단계별로 쌓이는 5성 한손검은?", "weapon:absolution"),
    ("w02", "적을 쓰러뜨릴 때 공격력이 일정 시간 강화되고 이 효과가 여러 번 중첩되는 4성 한손검은?", "weapon:blackclifflongsword"),
    ("w03", "강공격과 원소전투 스킬을 각각 적중시키면 원소 마스터리가 상승하는 4성 법구는?", "weapon:dawningfrost"),
    ("w04", "치명타가 발생했을 때 확률적으로 원소 구슬을 생성해 에너지를 회복시키는 4성 양손검은?", "weapon:favoniusgreatsword"),
    ("w05", "공격 속도 증가와 함께 일반 공격, 스킬, 강공격으로 전투 중 피해 증가 중첩을 쌓는 5성 양손검은?", "weapon:gestofthemightywolf"),
    ("w06", "불 또는 번개 원소의 영향을 받은 적을 상대할 때 추가 피해를 주는 4성 한손검은?", "weapon:lionsroar"),
    ("w07", "달빛 관련 원소 반응을 일으킬 때 에너지를 회복하고 해당 반응의 치명타 피해까지 높이는 5성 법구는?", "weapon:nocturnescurtaincall"),
    ("w08", "치유로 표식을 모은 뒤 원소전투 스킬이나 폭발을 사용하면 공격력과 원소 피해가 증가하는 4성 활은?", "weapon:rangegauge"),
    ("w09", "원소 반응 후 최대 HP가 증가하며 특정 달빛 상태에서는 HP 증가 효과가 더 커지는 4성 한손검은?", "weapon:serenityscall"),
    ("w10", "질주에 필요한 스태미나를 줄이고 대시 이후 일반 공격을 강화하는 4성 한손검은?", "weapon:sturdybone"),
    ("w11", "원소전투 스킬 사용 후 일반 공격이 강화되고 추가 원소 피해를 주는 5성 활은?", "weapon:astralvulturescrimsonplumage"),
    ("w12", "보호막이 있을 때 공격력이 상승하고 보호막 강도도 높아지는 4성 장병기는?", "weapon:rightfulreward"),
    ("a01", "한손검·양손검·장병기를 사용할 때 일반 공격 피해가 35% 증가하는 성유물은?", "artifact:gladiatorsfinale"),
    ("a02", "일반 공격이 치명타를 내면 강공격 피해가 증가하는 성유물은?", "artifact:bloodstainedchivalry"),
    ("a03", "원소 마스터리가 높을수록 과부하, 초전도, 감전, 연소 반응이 강해지는 성유물은?", "artifact:thunderingfury"),
    ("a04", "물 원소 영향을 받은 적에게 주는 피해가 증가하는 성유물은?", "artifact:heartofdepth"),
    ("a05", "얼음 원소 영향을 받은 적을 공격하면 치명타 확률이 증가하고 빙결 상태면 추가로 증가하는 성유물은?", "artifact:blizzardstrayer"),
    ("a06", "불 원소의 영향을 받은 적에게 주는 피해가 35% 증가하는 성유물은?", "artifact:lavawalker"),
    ("a07", "바위 원소 영향을 받은 적에게 주는 피해가 증가하는 성유물은?", "artifact:archaicpetra"),
    ("a08", "번개 원소 영향을 받은 적에게 주는 피해가 증가하는 성유물은?", "artifact:thundersoother"),
    ("a09", "풀 원소 영향을 받은 적에게 주는 피해가 증가하는 성유물은?", "artifact:deepwoodmemories"),
    ("a10", "바람 원소 영향을 받은 적에게 주는 피해가 증가하는 성유물은?", "artifact:viridescentvenerer"),
    ("a11", "법구나 활을 사용할 때 강공격 피해가 35% 증가하는 성유물은?", "artifact:wandererstroupe"),
    ("a12", "보호막이 있을 때 공격력이 상승하는 성유물은?", "artifact:retracingbolide"),
]

effect_questions = [
    {
        "id": eid,
        "question": q,
        "gold_docs": [gold],
        "kind": gold.split(":")[0],
        "qtype": "효과",
    }
    for eid, q, gold in effect_data
]
print(f"효과 질문: {len(effect_questions)}개")

all_questions = name_questions + nickname_questions + effect_questions
print(f"\n전체 평가 질문: {len(all_questions)}개\n")


# ============================================================
# 2. 평가 실행
# ============================================================

def evaluate(search_fn, qs):
    out = []
    for q in qs:
        retrieved = search_fn(q["question"]) or []
        retrieved = list(dict.fromkeys(retrieved))
        gold = q["gold_docs"][0]
        rank = retrieved.index(gold) + 1 if gold in retrieved else None
        out.append({
            "kind": q["kind"],
            "qtype": q["qtype"],
            "hit1": rank == 1,
            "hit5": rank is not None and rank <= 5,
        })
    return out

print("벡터 검색 평가 중...")
vector_results = evaluate(vector_search, all_questions)

print("챗봇 방식 평가 중...")
chatbot_results = evaluate(chatbot_search, all_questions)


# ============================================================
# 3. 표 만들기
# ============================================================

def summarize(results, method):
    rows = []
    groups = defaultdict(list)
    for r in results:
        groups[(r["kind"], r["qtype"])].append(r)

    order = [("weapon", "정식 이름"), ("weapon", "별명"), ("weapon", "효과"),
             ("artifact", "정식 이름"), ("artifact", "별명"), ("artifact", "효과")]

    for kind, qtype in order:
        items = groups.get((kind, qtype), [])
        if not items:
            continue
        n = len(items)
        h1 = sum(x["hit1"] for x in items)
        h5 = sum(x["hit5"] for x in items)
        rows.append({
            "방식": method,
            "대상": "무기" if kind == "weapon" else "성유물",
            "질문 유형": qtype,
            "문항 수": n,
            "Hit@1": f"{h1}/{n} ({h1/n*100:.1f}%)",
            "Hit@5": f"{h5}/{n} ({h5/n*100:.1f}%)",
        })
    return rows

table = []
table += summarize(vector_results, "벡터 검색")
table += summarize(chatbot_results, "챗봇 방식")

# 챗봇 전체
n = len(chatbot_results)
h1 = sum(r["hit1"] for r in chatbot_results)
h5 = sum(r["hit5"] for r in chatbot_results)
table.append({
    "방식": "챗봇 방식",
    "대상": "전체",
    "질문 유형": "전체",
    "문항 수": n,
    "Hit@1": f"{h1}/{n} ({h1/n*100:.1f}%)",
    "Hit@5": f"{h5}/{n} ({h5/n*100:.1f}%)",
})

df = pd.DataFrame(table)
print("\n" + "=" * 70)
print("최종 평가 결과 (레포 코드 기준)")
print("=" * 70)
display(df)


import json
from pathlib import Path

all_qs = name_questions + nickname_questions + effect_questions

out_path = Path("/content/genshin-build-alchemist/data/eval/item_questions_dev.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_qs, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path} ({len(all_qs)}개)")

정식 이름 질문: 12개 (무기 12, 성유물 0)
별명 질문: 12개
효과 질문: 24개

전체 평가 질문: 48개

벡터 검색 평가 중...
챗봇 방식 평가 중...

최종 평가 결과 (레포 코드 기준)


,방식,대상,질문 유형,문항 수,Hit@1,Hit@5
0,벡터 검색,무기,정식 이름,12,12/12 (100.0%),12/12 (100.0%)
1,벡터 검색,무기,별명,6,5/6 (83.3%),5/6 (83.3%)
2,벡터 검색,무기,효과,12,3/12 (25.0%),11/12 (91.7%)
3,벡터 검색,성유물,별명,6,4/6 (66.7%),4/6 (66.7%)
4,벡터 검색,성유물,효과,12,3/12 (25.0%),11/12 (91.7%)
5,챗봇 방식,무기,정식 이름,12,12/12 (100.0%),12/12 (100.0%)
6,챗봇 방식,무기,별명,6,6/6 (100.0%),6/6 (100.0%)
7,챗봇 방식,무기,효과,12,3/12 (25.0%),11/12 (91.7%)
8,챗봇 방식,성유물,별명,6,4/6 (66.7%),4/6 (66.7%)
9,챗봇 방식,성유물,효과,12,3/12 (25.0%),11/12 (91.7%)


저장 완료: /content/genshin-build-alchemist/data/eval/item_questions_dev.json (48개)


In [17]:
# 13-1) 성유물 정식 이름 질문 추가 + 최종 표 재생성

import re
import numpy as np
from collections import defaultdict
import pandas as pd

# ============================================================
# 1. 성유물 정식 이름 질문 생성
# ============================================================

# 성유물 후보 (4~5성 우선, 부족하면 3성 이하 채움)
artifact_candidates = []
for a in kb.artifacts:
    key = a.get("key")
    name = a.get("name")
    if not key or not name:
        continue
    doc_id = f"artifact:{key}"
    meta = metadata_by_id.get(doc_id, {})
    rarity = str(meta.get("rarity", ""))
    stars = set(re.findall(r"(?<!\d)([1-5])\s*성", rarity))
    is_high = bool(stars & {"4", "5"})
    artifact_candidates.append((doc_id, a, is_high))

high = [(d, m) for d, m, h in artifact_candidates if h]
low = [(d, m) for d, m, h in artifact_candidates if not h]

print(f"성유물 4~5성: {len(high)}개, 그 외: {len(low)}개")

def dedup(items):
    seen = set()
    out = []
    for doc_id, meta in items:
        norm = re.sub(r"[\W_]+", "", meta["name"]).casefold()
        if norm not in seen:
            seen.add(norm)
            out.append((doc_id, meta))
    return out

high = dedup(high)
low = dedup(low)

# 12개 채우기 (4~5성 우선)
selected_arts = high[:12]
if len(selected_arts) < 12:
    selected_arts += low[:12 - len(selected_arts)]

print(f"선택된 성유물: {len(selected_arts)}개")

# 성유물 정식 이름 질문 생성
artifact_name_questions = []
for i, (doc_id, meta) in enumerate(selected_arts, 1):
    artifact_name_questions.append({
        "id": f"aname-{i:02d}",
        "question": f"{meta['name']} 성유물 효과 알려줘",
        "gold_docs": [doc_id],
        "kind": "artifact",
        "qtype": "정식 이름",
    })

print(f"성유물 정식 이름 질문: {len(artifact_name_questions)}개")


# ============================================================
# 2. 기존 질문 + 성유물 정식 이름 합치기
# ============================================================

full_questions = name_questions + nickname_questions + effect_questions + artifact_name_questions
print(f"전체 질문: {len(full_questions)}개")


# ============================================================
# 3. 평가 재실행
# ============================================================

def evaluate(search_fn, qs):
    out = []
    for q in qs:
        retrieved = search_fn(q["question"]) or []
        retrieved = list(dict.fromkeys(retrieved))
        gold = q["gold_docs"][0]
        rank = retrieved.index(gold) + 1 if gold in retrieved else None
        out.append({
            "kind": q["kind"],
            "qtype": q["qtype"],
            "hit1": rank == 1,
            "hit5": rank is not None and rank <= 5,
        })
    return out

print("\n벡터 검색 평가 중...")
vector_results = evaluate(vector_search, full_questions)

print("챗봇 방식 평가 중...")
chatbot_results = evaluate(chatbot_search, full_questions)


# ============================================================
# 4. 최종 표
# ============================================================

def summarize(results, method):
    rows = []
    groups = defaultdict(list)
    for r in results:
        groups[(r["kind"], r["qtype"])].append(r)

    order = [
        ("weapon", "정식 이름"), ("weapon", "별명"), ("weapon", "효과"),
        ("artifact", "정식 이름"), ("artifact", "별명"), ("artifact", "효과"),
    ]
    for kind, qtype in order:
        items = groups.get((kind, qtype), [])
        if not items:
            continue
        n = len(items)
        h1 = sum(x["hit1"] for x in items)
        h5 = sum(x["hit5"] for x in items)
        rows.append({
            "방식": method,
            "대상": "무기" if kind == "weapon" else "성유물",
            "질문 유형": qtype,
            "문항 수": n,
            "Hit@1": f"{h1}/{n} ({h1/n*100:.1f}%)",
            "Hit@5": f"{h5}/{n} ({h5/n*100:.1f}%)",
        })
    return rows

table = []
table += summarize(vector_results, "벡터 검색")
table += summarize(chatbot_results, "챗봇 방식")

n = len(chatbot_results)
h1 = sum(r["hit1"] for r in chatbot_results)
h5 = sum(r["hit5"] for r in chatbot_results)
table.append({
    "방식": "챗봇 방식",
    "대상": "전체",
    "질문 유형": "전체",
    "문항 수": n,
    "Hit@1": f"{h1}/{n} ({h1/n*100:.1f}%)",
    "Hit@5": f"{h5}/{n} ({h5/n*100:.1f}%)",
})

df = pd.DataFrame(table)
print("\n" + "=" * 70)
print("최종 평가 결과 (성유물 정식 이름 포함)")
print("=" * 70)
display(df)


all_qs = name_questions + nickname_questions + effect_questions + artifact_name_questions

out_path = Path("/content/genshin-build-alchemist/data/eval/item_questions_dev.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_qs, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {len(all_qs)}개")

성유물 4~5성: 60개, 그 외: 3개
선택된 성유물: 12개
성유물 정식 이름 질문: 12개
전체 질문: 60개

벡터 검색 평가 중...
챗봇 방식 평가 중...

최종 평가 결과 (성유물 정식 이름 포함)


,방식,대상,질문 유형,문항 수,Hit@1,Hit@5
0,벡터 검색,무기,정식 이름,12,12/12 (100.0%),12/12 (100.0%)
1,벡터 검색,무기,별명,6,5/6 (83.3%),5/6 (83.3%)
2,벡터 검색,무기,효과,12,3/12 (25.0%),11/12 (91.7%)
3,벡터 검색,성유물,정식 이름,12,11/12 (91.7%),12/12 (100.0%)
4,벡터 검색,성유물,별명,6,4/6 (66.7%),4/6 (66.7%)
5,벡터 검색,성유물,효과,12,3/12 (25.0%),11/12 (91.7%)
6,챗봇 방식,무기,정식 이름,12,12/12 (100.0%),12/12 (100.0%)
7,챗봇 방식,무기,별명,6,6/6 (100.0%),6/6 (100.0%)
8,챗봇 방식,무기,효과,12,3/12 (25.0%),11/12 (91.7%)
9,챗봇 방식,성유물,정식 이름,12,12/12 (100.0%),12/12 (100.0%)


저장 완료: 60개
